# 🔐 Vuln Detector v3 — GraphCodeBERT **Full-FT** + **Gated-Attention MIL** (nhị phân)

Bản v3 sửa 4 nhóm vấn đề tìm ra khi mổ xẻ v2 (headline 0.939 nhưng bị bơm bởi synthetic + confound):

| Nhóm | v2 (cũ) | v3 (sửa) |
|---|---|---|
| **1. Đo trung thực** | `synthetic_hardneg` (AUC~0.999, chỉ ghi nhớ 11 pattern) nằm trong val/test → bơm headline | **synthetic CHỈ vào TRAIN**; val/test = dữ liệu THẬT. Báo cáo **per-source MODEL vs túi-từ** + **recall theo SWC** |
| **2. Chunking mất code** | `MAX_CHUNKS=8` → contract dài (dappscan) bị `linspace` vứt cửa sổ chứa lỗi (Safe 29% / Vuln 11% bị cắt) | **`MAX_CHUNKS=16`**, `TRAIN_BS=2`/`GRAD_ACCUM=8` → peak GPU **không đổi**, effective batch 16 giữ nguyên |
| **3. Data** | balanced độ dài **toàn cục** → smartbug Safe 4.6k vs Vuln 9.5k; nhãn dappscan lẫn SWC mờ | **`detect_dataset_balanced_v2.jsonl`**: cân bằng độ dài **theo nguồn** (len-AUC 0.65→0.52) + lọc SWC informational |
| **4. Kỹ thuật chết** | multi-task `aux_weight=0` (data không categories); class_weight ~[1,1] | **Bỏ hẳn nhánh multi-task**; ghi rõ class_weight ~no-op |

> Kaggle: **Add Data → upload `detect_dataset_balanced_v2.jsonl`** (chạy `clean_detect_dataset_v2.py` để sinh). Bật GPU. Chạy cell cài đặt → **Restart kernel** → Run All.
>
> ⚠️ Kỳ vọng thực tế: headline sẽ **THẤP hơn** 0.939 của v2 — vì đã bỏ phần điểm ảo. Điểm mới là điểm **thật**. Nhìn cột `Δ` (MODEL−BOW) per-source để biết model có hiểu hơn túi-từ không.

## ⚙️ 0. Cài đặt (ghim bản 4.x ổn định) — chạy rồi **RESTART KERNEL**

In [ ]:
# Env HF phải đặt TRƯỚC khi import transformers/huggingface_hub (tránh lỗi 403 Xet-CDN trên Kaggle)
import os
os.environ["HF_HUB_DISABLE_XET"]      = "1"   # route về LFS thường thay vì xet-bridge CDN
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0" # tắt hf_transfer cho ổn định

# Ghim bộ thư viện tương thích (tránh transformers 5.x lỗi trên Kaggle).
!pip install -q "transformers==4.46.3" "tokenizers==0.20.3" "huggingface_hub==0.25.2" "peft==0.13.2" "accelerate==1.0.1" "datasets==3.0.1" scikit-learn sentencepiece
import importlib
for pkg in ["transformers", "tokenizers", "huggingface_hub", "accelerate", "datasets", "torch"]:
    try:
        print(f"  {pkg}: {getattr(importlib.import_module(pkg), '__version__', '?')}")
    except Exception as e:
        print(f"  {pkg}: !! {e}")
print("\n🔴 BẮT BUỘC: Run -> Restart Kernel, rồi Run All. (transformers 5.x đã nạp sẵn -> phải restart.)")

## 📦 1. Cấu hình

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]  = "0"   # collator gộp-chunk-theo-contract KHÔNG hợp DataParallel -> 1 GPU
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json, re, glob, hashlib, random, collections
import numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F

import transformers
assert int(transformers.__version__.split(".")[0]) < 5, \
    f"transformers=={transformers.__version__}: CHƯA restart kernel! Restart rồi Run All."
print("transformers:", transformers.__version__)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ----------------------- HYPER-PARAMS -----------------------
MODEL_NAME     = "microsoft/graphcodebert-base"   # code-pretrained (dùng như encoder, KHÔNG dùng data-flow graph)
TOKENIZER_NAME = "roberta-base"                    # cùng vocab với graphcodebert -> nạp thẳng
MAX_LEN    = 512
# ── (Nhóm 2) Chunking: cap 16 để KHÔNG vứt code contract dài (dappscan) ──
#   Peak GPU ~ BS × MAX_CHUNKS. Giữ peak = train 2×16=32 / eval 4×16=64 == như cấu hình cũ (4×8 / 8×8).
MAX_CHUNKS = 16
LABEL2ID   = {"Safe": 0, "Vulnerable": 1}
ID2LABEL   = {0: "Safe", 1: "Vulnerable"}
SYNTH_SRC  = "synthetic_hardneg"   # (Nhóm 1) nguồn này CHỈ vào TRAIN, không vào val/test

# ── Capacity: FULL fine-tune (không LoRA) ──
USE_LORA        = False        # False = full fine-tune toàn bộ encoder
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.05   # chỉ dùng nếu USE_LORA=True
GRAD_CHECKPOINT = True

# ── Regularization (multi-task ĐÃ BỎ ở v3: data nhị phân không có categories -> nhánh đó chết) ──
ENT_REG          = 0.01        # phạt entropy attention -> dồn chú ý vào chunk chứa lỗi
LABEL_SMOOTHING  = 0.05
USE_CLASS_WEIGHT = True         # LƯU Ý: data đã cân bằng 50/50 -> class_weight ~[1,1], gần như no-op (giữ cho an toàn)

# ── Train ── (full-FT dùng LR NHỎ) ──
#   OOM? -> giảm MAX_CHUNKS 16->12, hoặc EVAL_BS 4->2. Giữ TRAIN_BS×GRAD_ACCUM = 16 (effective batch).
EPOCHS     = 10
TRAIN_BS   = 2      # (cũ 4) -> peak train = 2×16 = 32, bằng cấu hình cũ 4×8
EVAL_BS    = 4      # (cũ 8) -> peak eval  = 4×16 = 64, bằng cấu hình cũ 8×8
GRAD_ACCUM = 8      # (cũ 4) -> effective batch = 2×8 = 16, KHÔNG đổi động lực học
LR         = 2e-5 if not USE_LORA else 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
EARLY_STOP_PATIENCE = 3

OUTPUT_DIR = "./vuln-v3"
SAVE_DIR   = "./vuln-v3-final"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print(f"Model: {MODEL_NAME} | full_finetune={not USE_LORA} | LR={LR} | MAX_CHUNKS={MAX_CHUNKS} "
      f"| eff_batch={TRAIN_BS*GRAD_ACCUM}")

## 📂 2. Nạp dữ liệu (`detect_dataset_balanced_v2.jsonl`) — giữ `swc_ids` cho báo cáo per-SWC

In [ ]:
def find_data_path():
    # ƯU TIÊN v2 (lọc SWC mờ + cân bằng độ dài THEO NGUỒN + giữ swc_ids). Dataset trên Kaggle:
    #   /kaggle/input/datasets/thanhphuocjr/detect-dataset-balanced-v2/detect_dataset_balanced_v2.jsonl
    v2_pats = ["/kaggle/input/**/detect-dataset-balanced-v2/**/*.jsonl",  # bất kỳ .jsonl trong dataset v2
               "/kaggle/input/**/detect_dataset_balanced_v2.jsonl",       # đúng tên file, mọi độ sâu
               "detect_dataset_balanced_v2.jsonl",
               "../data/clean_dataset/detect_dataset_balanced_v2.jsonl",
               "data/clean_dataset/detect_dataset_balanced_v2.jsonl"]
    for pat in v2_pats:
        hits = sorted(glob.glob(pat, recursive=True))
        if hits: return hits[0], True
    # fallback: bản balanced CŨ (không có swc_ids) -> CẢNH BÁO TO để không train nhầm data cũ
    for pat in ["/kaggle/input/**/detect_dataset_balanced.jsonl", "detect_dataset_balanced.jsonl",
                "../data/clean_dataset/detect_dataset_balanced.jsonl",
                "data/clean_dataset/detect_dataset_balanced.jsonl"]:
        hits = sorted(glob.glob(pat, recursive=True))
        if hits: return hits[0], False
    raise FileNotFoundError("Không thấy data. Hãy Add Data: dataset 'detect-dataset-balanced-v2' "
                            "(chứa detect_dataset_balanced_v2.jsonl) lên Kaggle.")

DATA_PATH, IS_V2 = find_data_path()
print("DATA_PATH =", DATA_PATH, "| v2:", IS_V2)
if not IS_V2:
    print("\n" + "!"*70 + "\n⚠️  ĐANG DÙNG DATA CŨ (không phải v2)! Per-SWC report sẽ trống, confound độ dài\n"
          "    còn nguyên. Kiểm tra lại Add Data -> 'detect-dataset-balanced-v2'.\n" + "!"*70)

rows = [json.loads(l) for l in open(DATA_PATH, encoding="utf-8") if l.strip()]
rows = [r for r in rows if r.get("code") and r.get("label") in LABEL2ID]
for r in rows:
    r.setdefault("source", "unknown"); r.setdefault("swc_ids", [])

n, nv = len(rows), sum(r["label"] == "Vulnerable" for r in rows)
HAS_SWC = any(r.get("swc_ids") for r in rows)
print(f"Mẫu: {n:,} | Vuln {nv:,} ({100*nv/n:.1f}%) | Safe {n-nv:,} | có swc_ids: {HAS_SWC}")
print("Theo nguồn:", dict(collections.Counter(r["source"] for r in rows)))

# Nguồn KHÔNG còn là proxy của nhãn (mỗi nguồn nên có cả 2 lớp sau cân bằng-theo-nguồn)
_sl = collections.defaultdict(lambda: [0, 0])
for r in rows: _sl[r["source"]][1 if r["label"] == "Vulnerable" else 0] += 1
print("Nguồn x nhãn (Safe, Vuln):", {k: tuple(v) for k, v in _sl.items()})

# Phân bố SWC (để nhóm 1 báo cáo per-SWC ở cell đánh giá)
if HAS_SWC:
    swc_cnt = collections.Counter(s for r in rows if r["label"] == "Vulnerable" for s in (r.get("swc_ids") or []))
    print("Top SWC (Vuln):", swc_cnt.most_common(12))

## ✂️ 3. Chia tập CHỐNG RÒ RỈ near-duplicate + **synthetic chỉ vào TRAIN**

**Chống rò rỉ (giữ như v2):** gộp **(a)** clone cấu trúc + **(b)** near-dup `cosine ≥ 0.85` vào cùng một *cụm*, rồi cho **cả cụm về đúng 1 split** → không thể rò rỉ. Kèm kiểm chứng rò rỉ còn sót (kỳ vọng ~0).

**Mới ở v3 (Nhóm 1):** mọi cụm chứa `synthetic_hardneg` bị **ép vào TRAIN** → val/test **chỉ còn dữ liệu THẬT**. Lý do: synthetic tách nhãn được chỉ bằng ~11 chuỗi anti-pattern cố định (AUC~0.999) → nếu để trong test sẽ **bơm headline giả**. Baseline túi-từ giờ đo trên **test thật**, làm *trần lối tắt* để biết GraphCodeBERT có hiểu hơn bag-of-words không.

In [ ]:
# ✂️ 3. Chia tập CHỐNG RÒ RỈ 2 lớp -> cả cụm gần-trùng về đúng 1 split (không thể rò rỉ)
#    (a) chữ ký cấu trúc (đổi định danh -> X): bắt clone đổi-tên-biến
#    (b) near-duplicate: TF-IDF word 4-6gram + cosine >= ngưỡng
#    (Nhóm 1) synthetic_hardneg CHỈ vào TRAIN -> val/test là DỮ LIỆU THẬT, headline không bị bơm bởi AUC~0.999
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score as _f1m, roc_auc_score as _aucm

NEAR_DUP_SIM     = 0.85   # cosine >= mức này coi là gần-trùng -> ép cùng 1 split
EVAL_CLUSTER_CAP = 30     # cụm gần-trùng lớn hơn mức này -> ép vào TRAIN (giữ val/test đa dạng)

n      = len(rows)
codes  = [r["code"] for r in rows]
y_all  = np.array([1 if r["label"] == "Vulnerable" else 0 for r in rows])
is_syn = np.array([r.get("source") == SYNTH_SRC for r in rows])

# ---------- Union-Find ----------
parent = list(range(n))
def _find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def _union(a, b):
    ra, rb = _find(a), _find(b)
    if ra != rb: parent[ra] = rb

# (a) gộp clone cấu trúc (đổi mọi định danh -> X rồi hash)
def structural_sig(code):
    s = re.sub(r"\b[A-Za-z_]\w*\b", "X", code)
    return hashlib.md5(re.sub(r"\s+", "", s).encode("utf-8", "replace")).hexdigest()
sig_groups = collections.defaultdict(list)
for i, c in enumerate(codes): sig_groups[structural_sig(c)].append(i)
for g in sig_groups.values():
    for j in g[1:]: _union(g[0], j)

# (b) gộp near-duplicate qua TF-IDF cosine (chặn theo khối cho đỡ tốn RAM)
vec_dup = TfidfVectorizer(analyzer="word", ngram_range=(4, 6), min_df=2,
                          max_features=120000, dtype=np.float32)
Xd = vec_dup.fit_transform(codes)          # TF-IDF đã L2-normalize -> Xd @ Xd.T = cosine
BLK, n_pairs = 512, 0
for s in range(0, n, BLK):
    sim = Xd[s:s+BLK] @ Xd.T
    sim.data[sim.data < NEAR_DUP_SIM] = 0.0
    sim.eliminate_zeros(); sim = sim.tocoo()
    for li, j in zip(sim.row, sim.col):
        gi = s + int(li)
        if gi < int(j): _union(gi, int(j)); n_pairs += 1

# ---------- Gộp thành cụm & chia theo cụm ----------
comp = collections.defaultdict(list)
for i in range(n): comp[_find(i)].append(i)
clusters = list(comp.values()); random.Random(SEED).shuffle(clusters)
big = sum(1 for c in clusters if len(c) > 1); mx_c = max(len(c) for c in clusters)
print(f"Gộp near-dup: {n_pairs:,} cặp cosine>={NEAR_DUP_SIM}")
print(f"{n:,} mẫu -> {len(clusters):,} cụm độc lập | {big:,} cụm gần-trùng(>1) | cụm lớn nhất {mx_c}")

# val/test rút từ DỮ LIỆU THẬT (không synthetic); size ~10% của phần thật
n_real = int((~is_syn).sum())
nt, nvl = int(0.10 * n_real), int(0.10 * n_real)
test_idx, val_idx, train_idx = [], [], []
for cl in clusters:
    cluster_has_syn = any(is_syn[i] for i in cl)
    if   cluster_has_syn:                                                        train_idx += cl   # synthetic -> TRAIN
    elif len(cl) <= EVAL_CLUSTER_CAP and len(test_idx) + len(cl) <= nt:          test_idx  += cl
    elif len(cl) <= EVAL_CLUSTER_CAP and len(val_idx)  + len(cl) <= nvl:         val_idx   += cl
    else:                                                                        train_idx += cl
take = lambda ix: [rows[i] for i in ix]
train_raw, val_raw, test_raw = take(train_idx), take(val_idx), take(test_idx)
print(f"(Nhóm 1) synthetic CHỈ vào train: {int(is_syn.sum())} mẫu | val/test = dữ liệu THẬT")

def bal(nm, d):
    v = sum(x["label"] == "Vulnerable" for x in d)
    bysrc = dict(collections.Counter(x.get("source", "?") for x in d))
    print(f"  {nm:5s}: {len(d):5,d} | Vuln {v:5,d} ({100*v/max(1,len(d)):.1f}%)  Safe {len(d)-v:5,d} | {bysrc}")
bal("train", train_raw); bal("val", val_raw); bal("test", test_raw)

# ---------- KIỂM CHỨNG rò rỉ còn sót: max cosine mỗi mẫu val/test -> TRAIN (kỳ vọng ~0) ----------
Xtr = Xd[train_idx]
def _residual(name, idx):
    if not idx: return
    Xq = Xd[idx]; mxs = np.zeros(len(idx), dtype=np.float32)
    for s in range(0, len(idx), 512):
        mxs[s:s+512] = (Xq[s:s+512] @ Xtr.T).max(axis=1).toarray().ravel()
    print(f"  {name}: {int((mxs >= NEAR_DUP_SIM).sum())}/{len(idx)} mẫu còn cosine>={NEAR_DUP_SIM} tới train"
          f" | median max-sim={np.median(mxs):.2f}")
print("Rò rỉ còn sót:"); _residual("val ", val_idx); _residual("test", test_idx)

# ---------- TRẦN LỐI TẮT: baseline túi-từ (unigram+bigram) trên CHÍNH split sạch (test THẬT) ----------
vec_bow = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=3,
                          max_features=40000, sublinear_tf=True, dtype=np.float32)
Xb  = vec_bow.fit_transform(codes)
bow_clf = LogisticRegression(max_iter=2000, C=4.0, class_weight="balanced")
bow_clf.fit(Xb[train_idx], y_all[train_idx])
bow_test = bow_clf.predict_proba(Xb[test_idx])[:, 1]   # dùng lại ở cell 7 để so per-source
print(f"\n[BASELINE túi-từ | test THẬT] F1m={_f1m(y_all[test_idx], (bow_test>=0.5).astype(int), average='macro'):.3f}"
      f" AUC={_aucm(y_all[test_idx], bow_test):.3f}  <-- GraphCodeBERT phải VƯỢT rõ mốc này mới là 'hiểu' hơn túi-từ")

## 🪟 4. Tokenizer & Chunk hoá (contract ≤ MAX_CHUNKS giữ liền mạch — không mất code)

In [ ]:
import subprocess, sys
for _p in ["sentencepiece", "tiktoken"]:
    try: importlib.import_module(_p)
    except Exception: subprocess.run([sys.executable, "-m", "pip", "install", "-q", _p], check=False)

from transformers import AutoTokenizer
def load_tokenizer(name):
    try: return AutoTokenizer.from_pretrained(name, use_fast=True)
    except Exception as e:
        print("Fast lỗi -> slow:", str(e)[:100]); return AutoTokenizer.from_pretrained(name, use_fast=False)
tokenizer = load_tokenizer(TOKENIZER_NAME)
for attr, tk in [("bos_token","<s>"),("eos_token","</s>"),("unk_token","<unk>"),
                 ("pad_token","<pad>"),("cls_token","<s>"),("sep_token","</s>"),("mask_token","<mask>")]:
    if getattr(tokenizer, attr, None) is None:
        try: setattr(tokenizer, attr, tk)
        except Exception: pass
CLS = tokenizer.convert_tokens_to_ids("<s>"); SEP = tokenizer.convert_tokens_to_ids("</s>")
PAD = tokenizer.convert_tokens_to_ids("<pad>"); CONTENT = MAX_LEN - 2
assert all(isinstance(x, int) and x >= 0 for x in (CLS, SEP, PAD))
print(f"Tokenizer OK | vocab={tokenizer.vocab_size:,} | CLS/SEP/PAD={CLS}/{SEP}/{PAD}")

def chunk_encode(code):
    # (Nhóm 2) contract <= MAX_CHUNKS -> giữ LIỀN MẠCH (không mất code).
    #          contract > MAX_CHUNKS (rất hiếm) -> lấy mẫu cách đều để phủ toàn contract thay vì cắt cụt đuôi.
    ids = tokenizer.encode(code, add_special_tokens=False) or [PAD]
    wins = [ids[i:i+CONTENT] for i in range(0, len(ids), CONTENT)]
    if len(wins) > MAX_CHUNKS:
        sel = sorted(set(np.linspace(0, len(wins)-1, MAX_CHUNKS).round().astype(int).tolist()))
        wins = [wins[i] for i in sel]
    iid, am = [], []
    for w in wins:
        seq = [CLS] + w + [SEP]; m = [1]*len(seq)
        if len(seq) < MAX_LEN:
            p = MAX_LEN - len(seq); seq += [PAD]*p; m += [0]*p
        iid.append(seq); am.append(m)
    return torch.tensor(iid, dtype=torch.long), torch.tensor(am, dtype=torch.long)

class ChunkedDS(torch.utils.data.Dataset):
    def __init__(self, items):
        self.data = []
        for it in items:
            iid, am = chunk_encode(it["code"])
            self.data.append({"input_ids": iid, "attention_mask": am, "label": LABEL2ID[it["label"]]})
    def __len__(self): return len(self.data)
    def __getitem__(self, i): return self.data[i]

print("Đang chunk hoá...")
train_ds, val_ds, test_ds = ChunkedDS(train_raw), ChunkedDS(val_raw), ChunkedDS(test_raw)
tc = sum(d["input_ids"].shape[0] for d in train_ds.data)
mxc = max(d["input_ids"].shape[0] for d in train_ds.data)
print(f"Train: {len(train_ds):,} contract -> {tc:,} chunk (TB {tc/len(train_ds):.2f}/contract, max {mxc})")

def collate(fs):
    return {"input_ids":      torch.cat([f["input_ids"] for f in fs], 0),
            "attention_mask": torch.cat([f["attention_mask"] for f in fs], 0),
            "n_chunks":       torch.tensor([f["input_ids"].shape[0] for f in fs], dtype=torch.long),
            "labels":         torch.tensor([f["label"] for f in fs], dtype=torch.long)}

## 🤖 5. Mô hình: Full fine-tune encoder + Gated-Attention MIL pooling (1 nhánh nhị phân)

In [ ]:
from transformers import AutoModel
from transformers.modeling_outputs import SequenceClassifierOutput
from peft import LoraConfig, get_peft_model, TaskType

class HierMIL(nn.Module):
    """Full-FT encoder + Gated-Attention MIL pooling theo contract, 1 nhánh nhị phân.
    (v3: đã BỎ nhánh loại-lỗ-hổng vì data nhị phân không có categories -> nhánh đó vô tác dụng)."""
    def __init__(self, encoder, hidden, dropout=0.1, class_weights=None,
                 label_smoothing=0.0, ent_reg=0.0):
        super().__init__()
        self.encoder = encoder
        self.attn_V = nn.Linear(hidden, 128); self.attn_U = nn.Linear(hidden, 128)
        self.attn_w = nn.Linear(128, 1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, 2)          # nhánh nhị phân
        self.label_smoothing = label_smoothing
        self.ent_reg = ent_reg
        if class_weights is not None: self.register_buffer("class_weights", class_weights)
        else: self.class_weights = None

    def forward(self, input_ids, attention_mask, n_chunks, labels=None):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        A = self.attn_w(torch.tanh(self.attn_V(h)) * torch.sigmoid(self.attn_U(h)))   # [N,1]
        pooled, ent, idx = [], h.new_zeros(()), 0
        for c in n_chunks:
            c = int(c); ai = torch.softmax(A[idx:idx+c], dim=0)
            pooled.append((ai * h[idx:idx+c]).sum(0))
            if c > 1:
                p = ai.squeeze(-1).clamp_min(1e-9); ent = ent + -(p * torch.log(p)).sum()
            idx += c
        z = self.dropout(torch.stack(pooled, 0))          # [B,H]
        bin_logits = self.classifier(z)
        loss = None
        if labels is not None:
            w = self.class_weights if self.class_weights is not None else None
            loss = F.cross_entropy(bin_logits, labels, weight=w, label_smoothing=self.label_smoothing)
            if self.ent_reg > 0:
                loss = loss + self.ent_reg * (ent / max(1, len(n_chunks)))
        return SequenceClassifierOutput(loss=loss, logits=bin_logits)

# --- Tải encoder CÓ RETRY + fallback mirror (Kaggle hay lỗi 403 Xet-CDN với pytorch_model.bin) ---
import time, glob as _glob, huggingface_hub
def _set_endpoint(url):
    huggingface_hub.constants.ENDPOINT = url
    try: huggingface_hub.file_download.ENDPOINT = url
    except Exception: pass
    os.environ["HF_ENDPOINT"] = url

def load_encoder(name):
    # 0) Nếu đã Add Data model offline (dataset chứa config.json + pytorch_model.bin) -> nạp thẳng, KHỎI mạng
    for pat in ["/kaggle/input/**/graphcodebert*/**/config.json", "/kaggle/input/**/graphcodebert*/config.json"]:
        hit = sorted(_glob.glob(pat, recursive=True))
        if hit:
            local = os.path.dirname(hit[0]); print(f"  Nạp OFFLINE từ: {local}")
            return AutoModel.from_pretrained(local, trust_remote_code=True, local_files_only=True)
    # 1) Online: thử endpoint chính 3 lần (mỗi lần chữ ký CDN mới), rồi mirror 3 lần
    attempts = []
    for endpoint in ["https://huggingface.co", "https://hf-mirror.com"]:
        _set_endpoint(endpoint)
        for k in range(3):
            try:
                print(f"  tải {name} | endpoint={endpoint} | thử {k+1}/3 ...")
                return AutoModel.from_pretrained(name, trust_remote_code=True, force_download=(k > 0))
            except Exception as e:
                attempts.append(f"{endpoint} #{k+1}: {str(e)[:110]}")
                print(f"    lỗi: {str(e)[:110]}"); time.sleep(6)
    raise RuntimeError("❌ Không tải được weights sau nhiều lần thử:\n  " + "\n  ".join(attempts) +
                       "\n\n>> PHƯƠNG ÁN OFFLINE: tạo Kaggle Dataset chứa graphcodebert-base "
                       "(config.json + pytorch_model.bin + vocab/merges), Add Data, rồi Run lại. "
                       "Loader sẽ tự phát hiện và nạp offline.")

encoder = load_encoder(MODEL_NAME)
HIDDEN = encoder.config.hidden_size
if USE_LORA:
    encoder = get_peft_model(encoder, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
        task_type=TaskType.FEATURE_EXTRACTION,
        target_modules=["query","key","value","output.dense","intermediate.dense"]))
    encoder.print_trainable_parameters()
if GRAD_CHECKPOINT:
    try:
        encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        if hasattr(encoder, "enable_input_require_grads"): encoder.enable_input_require_grads()
        print("Gradient checkpointing: ON")
    except Exception as e:
        print("Grad ckpt OFF:", e)

cnt = collections.Counter(d["label"] for d in train_ds.data); N = sum(cnt.values())
cw = torch.tensor([N/(2*cnt[0]), N/(2*cnt[1])], dtype=torch.float) if USE_CLASS_WEIGHT else None
model = HierMIL(encoder, HIDDEN, dropout=0.1, class_weights=cw,
                label_smoothing=LABEL_SMOOTHING, ent_reg=ENT_REG).to(device)
print(f"✅ Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} "
      f"| full_FT={not USE_LORA} | ent_reg={ENT_REG} | class_weights={cw}")

## 🏋️ 6. Huấn luyện

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import f1_score

def _lg(p): return p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
def compute_metrics(p):
    pr = np.argmax(_lg(p), axis=-1)
    return {"f1_macro": f1_score(p.label_ids, pr, average="macro"),
            "f1_vuln":  f1_score(p.label_ids, pr, pos_label=1, average="binary", zero_division=0)}

args = TrainingArguments(
    output_dir=OUTPUT_DIR, num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR, weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO, lr_scheduler_type="cosine", fp16=torch.cuda.is_available(),
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="f1_macro", greater_is_better=True, logging_steps=50,
    save_total_limit=2, report_to="none", seed=SEED, remove_unused_columns=False,
    dataloader_pin_memory=False, label_names=["labels"], max_grad_norm=1.0,
    save_safetensors=False)   # model là nn.Module tuỳ chỉnh -> dùng torch.save cho an toàn

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=collate, compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)])
print("🚀 Train..."); trainer.train(); print("✅ Xong.")

# --- Đường cong train/val ---
print("\nEpoch |  train_loss |  eval_loss | f1_macro | f1_vuln")
tr = {}
for h in trainer.state.log_history:
    if "loss" in h and "epoch" in h and "eval_loss" not in h:
        tr[round(h["epoch"])] = h["loss"]
for h in trainer.state.log_history:
    if "eval_f1_macro" in h:
        e = round(h["epoch"])
        print(f"  {e:3d} | {tr.get(e, float('nan')):10.4f} | {h['eval_loss']:9.4f} | "
              f"{h['eval_f1_macro']:.4f} | {h['eval_f1_vuln']:.4f}")

## 📊 7. Đánh giá TRUNG THỰC: headline (test thật) + **per-source MODEL vs túi-từ** + **recall theo SWC**

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score, f1_score, accuracy_score)

def get_probs(ds):
    out = trainer.predict(ds)
    lg = out.predictions[0] if isinstance(out.predictions, tuple) else out.predictions
    return torch.softmax(torch.tensor(lg), dim=-1)[:, 1].numpy(), out.label_ids

# ---------- Ngưỡng tối ưu trên VAL (dữ liệu THẬT) ----------
val_probs, val_labels = get_probs(val_ds)
best_t, best_f1 = 0.5, -1.0
for t in np.linspace(0.05, 0.95, 91):
    f1m = f1_score(val_labels, (val_probs >= t).astype(int), average="macro")
    if f1m > best_f1: best_f1, best_t = f1m, float(t)
print(f"Ngưỡng tối ưu (VAL) = {best_t:.3f} | F1-macro(val) = {best_f1:.4f}")

# ---------- TEST (dữ liệu THẬT: KHÔNG có synthetic -> headline TRUNG THỰC) ----------
test_probs, test_labels = get_probs(test_ds)
for nm, t in [("0.50", 0.5), (f"{best_t:.3f} (hiệu chỉnh)", best_t)]:
    preds = (test_probs >= t).astype(int)
    print("\n" + "="*60 + f"\n📊 TEST @ threshold {nm}\n" + "="*60)
    print(classification_report(test_labels, preds, target_names=["Safe","Vulnerable"], digits=4, zero_division=0))
    print("Confusion [[TN FP][FN TP]]:\n", confusion_matrix(test_labels, preds))
model_auc = roc_auc_score(test_labels, test_probs)
print(f"\n>> HEADLINE (test THẬT, n={len(test_labels)}): "
      f"acc={accuracy_score(test_labels,(test_probs>=best_t).astype(int)):.3f} "
      f"F1m={f1_score(test_labels,(test_probs>=best_t).astype(int),average='macro'):.3f} "
      f"ROC-AUC={model_auc:.4f} PR-AUC={average_precision_score(test_labels, test_probs):.4f}")
print(f"   (so trần túi-từ ở cell 3: model {'VƯỢT' if model_auc>_aucm(y_all[test_idx],bow_test) else 'KHÔNG vượt'} "
      f"BOW AUC={_aucm(y_all[test_idx],bow_test):.4f})")

# ---------- Per-source: MODEL vs BOW (nguồn nào model thật sự hiểu hơn túi-từ?) ----------
preds = (test_probs >= best_t).astype(int)
src = np.array([r.get("source", "unknown") for r in test_raw])
print("\n--- Điểm theo NGUỒN (test THẬT): MODEL vs BASELINE túi-từ ---")
print(f"{'source':16s} {'n':>5s} {'acc':>7s} {'f1m':>7s} {'AUC_model':>10s} {'AUC_bow':>8s} {'Δ':>7s}  note")
for s in sorted(set(src)):
    ix = np.where(src == s)[0]
    yt = test_labels[ix]
    two = len(set(yt)) > 1
    a_m = roc_auc_score(yt, test_probs[ix]) if two else float("nan")
    a_b = roc_auc_score(yt, bow_test[ix])   if two else float("nan")
    note = "" if two else "⚠️ 1-lớp"
    print(f"{s:16s} {len(ix):5d} {accuracy_score(yt, preds[ix]):7.3f} "
          f"{f1_score(yt, preds[ix], average='macro'):7.3f} {a_m:10.3f} {a_b:8.3f} {a_m-a_b:+7.3f}  {note}")

# ---------- Per-SWC: model BẮT được loại lỗ hổng nào? (recall trên mẫu Vuln, @ngưỡng hiệu chỉnh) ----------
if HAS_SWC:
    vuln_ix = np.where(test_labels == 1)[0]
    by_swc = collections.defaultdict(list)
    for i in vuln_ix:
        for swc in (test_raw[i].get("swc_ids") or ["(no-swc)"]):
            by_swc[swc].append(int(preds[i] == 1))
    print("\n--- RECALL theo SWC (chỉ mẫu Vuln, ngưỡng hiệu chỉnh) — loại nào model hay BỎ SÓT ---")
    print(f"{'SWC':10s} {'n':>4s} {'recall':>7s}")
    for swc, hits in sorted(by_swc.items(), key=lambda kv: -len(kv[1])):
        if len(hits) >= 5:   # bỏ SWC quá hiếm cho khỏi nhiễu
            print(f"{swc:10s} {len(hits):4d} {np.mean(hits):7.3f}")

## 💾 8. Lưu mô hình

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
torch.save(model.state_dict(), os.path.join(SAVE_DIR, "model_state.pt"))
tokenizer.save_pretrained(SAVE_DIR)
json.dump({"model_name": MODEL_NAME, "tokenizer_name": TOKENIZER_NAME, "max_len": MAX_LEN,
           "max_chunks": MAX_CHUNKS, "hidden": HIDDEN, "use_lora": USE_LORA,
           "arch": "HierMIL", "threshold": best_t, "label2id": LABEL2ID},
          open(os.path.join(SAVE_DIR, "infer_config.json"), "w"), indent=2)
print("Đã lưu:", os.listdir(SAVE_DIR))

## 🔍 9. Inference thử

In [ ]:
model.eval()
@torch.no_grad()
def predict_contract(code, threshold=None):
    threshold = best_t if threshold is None else threshold
    iid, am = chunk_encode(code)
    out = model(input_ids=iid.to(device), attention_mask=am.to(device),
                n_chunks=torch.tensor([iid.shape[0]]).to(device))
    p = torch.softmax(out.logits, dim=-1)[0, 1].item()
    return ("Vulnerable" if p >= threshold else "Safe"), p

TEST_CODE = """
function withdraw(uint amount) public {
    require(balances[msg.sender] >= amount);
    (bool success,) = msg.sender.call{value: amount}("");
    require(success);
    balances[msg.sender] -= amount;   // giảm số dư SAU khi gửi -> reentrancy
}
"""
lbl, prob = predict_contract(TEST_CODE)
print(f"Dự đoán: {lbl}  (P(Vulnerable)={prob:.3f}, ngưỡng={best_t:.3f})")